In [34]:
import numpy as np
import pandas as pd 
import nltk
import os

In [35]:
train=pd.read_csv('train.txt',sep=';', names=['text', 'emotion'], header=None, encoding='ISO_8859-1')
train

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [36]:
train['emotion'].value_counts()
#columns to be oversampled anger,fear,love,surprise

emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

In [37]:
test=pd.read_csv('test.txt',sep=';',names=['text','emotion'],header=None,encoding='ISO_8859-1')
test.emotion.value_counts()

emotion
joy         695
sadness     581
anger       275
fear        224
love        159
surprise     66
Name: count, dtype: int64

In [38]:
df=pd.concat((train,test),ignore_index=True)

In [39]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(sparse_output=False)
emot_enc = enc.fit_transform(df[['emotion']])

encoded_df = pd.DataFrame(
    emot_enc,
    columns=enc.get_feature_names_out(['emotion'])
)

df1 = pd.concat([df.drop('emotion', axis=1), encoded_df], axis=1)


In [40]:
df1=df1.rename(columns={'emotion_anger': 'anger','emotion_fear': 'fear','emotion_joy': 'joy',
    'emotion_love': 'love','emotion_sadness': 'sadness','emotion_surprise': 'surprise'})

df1

,text,anger,fear,joy,love,sadness,surprise
0,i didnt feel humiliated,0.0,0.0,0.0,0.0,1.0,0.0
1,i can go from feeling so hopeless to so damned...,0.0,0.0,0.0,0.0,1.0,0.0
2,im grabbing a minute to post i feel greedy wrong,1.0,0.0,0.0,0.0,0.0,0.0
3,i am ever feeling nostalgic about the fireplac...,0.0,0.0,0.0,1.0,0.0,0.0
4,i am feeling grouchy,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
17995,i just keep feeling like someone is being unki...,1.0,0.0,0.0,0.0,0.0,0.0
17996,im feeling a little cranky negative after this...,1.0,0.0,0.0,0.0,0.0,0.0
17997,i feel that i am useful to my people and that ...,0.0,0.0,1.0,0.0,0.0,0.0
17998,im feeling more comfortable with derby i feel ...,0.0,0.0,1.0,0.0,0.0,0.0


In [41]:
label=['anger','fear','joy','love','sadness']

In [42]:
df1.drop(['surprise'],axis=1,inplace=True)

In [43]:
joy = df1[df1['joy'] == 1].sample(2000, random_state=42).index

df1 = df1.drop(joy).reset_index(drop=True)


In [44]:
df1 = df1[df1[label].sum(axis=1) > 0].reset_index(drop=True)

In [45]:
df1[label].value_counts()

anger  fear  joy  love  sadness
0.0    0.0   0.0  0.0   1.0        5247
             1.0  0.0   0.0        4057
1.0    0.0   0.0  0.0   0.0        2434
0.0    1.0   0.0  0.0   0.0        2161
       0.0   0.0  1.0   0.0        1463
Name: count, dtype: int64

In [46]:
df1.isna().sum()

text       0
anger      0
fear       0
joy        0
love       0
sadness    0
dtype: int64

In [47]:
text=df1['text']
text

0                                  i didnt feel humiliated
1        i can go from feeling so hopeless to so damned...
2         im grabbing a minute to post i feel greedy wrong
3        i am ever feeling nostalgic about the fireplac...
4                                     i am feeling grouchy
                               ...                        
15357                             i can feel its suffering
15358    i just keep feeling like someone is being unki...
15359    im feeling a little cranky negative after this...
15360    im feeling more comfortable with derby i feel ...
15361    i feel all weird when i have to meet w people ...
Name: text, Length: 15362, dtype: object

In [48]:
from nltk.tokenize import TweetTokenizer
tok=TweetTokenizer()
text=text.apply(lambda x:tok.tokenize(x)).apply(lambda x:" ".join(x))
text

0                                  i didnt feel humiliated
1        i can go from feeling so hopeless to so damned...
2         im grabbing a minute to post i feel greedy wrong
3        i am ever feeling nostalgic about the fireplac...
4                                     i am feeling grouchy
                               ...                        
15357                             i can feel its suffering
15358    i just keep feeling like someone is being unki...
15359    im feeling a little cranky negative after this...
15360    im feeling more comfortable with derby i feel ...
15361    i feel all weird when i have to meet w people ...
Name: text, Length: 15362, dtype: object

In [49]:
text=text.str.replace('[^A-Za-z0-9]'," ",regex=True)
text

0                                  i didnt feel humiliated
1        i can go from feeling so hopeless to so damned...
2         im grabbing a minute to post i feel greedy wrong
3        i am ever feeling nostalgic about the fireplac...
4                                     i am feeling grouchy
                               ...                        
15357                             i can feel its suffering
15358    i just keep feeling like someone is being unki...
15359    im feeling a little cranky negative after this...
15360    im feeling more comfortable with derby i feel ...
15361    i feel all weird when i have to meet w people ...
Name: text, Length: 15362, dtype: object

In [50]:
from nltk.stem import SnowballStemmer
st=SnowballStemmer('english')
text=text.apply(lambda x:[st.stem(i) for i in tok.tokenize(x) if len(i)>2]).apply(lambda x:" ".join(x))
text

0                                        didnt feel humili
1        can from feel hopeless damn hope just from be ...
2                        grab minut post feel greedi wrong
3        ever feel nostalg about the fireplac will know...
4                                             feel grouchi
                               ...                        
15357                                   can feel it suffer
15358    just keep feel like someon be unkind and do wr...
15359    feel littl cranki negat after this doctor appoint
15360    feel more comfort with derbi feel though can s...
15361    feel all weird when have meet peopl text but l...
Name: text, Length: 15362, dtype: object

In [51]:
from nltk.corpus import stopwords
words=stopwords.words('english')
text=text.apply(lambda x:[i for i in tok.tokenize(x) if i not in words]).apply(lambda x:" ".join(x))
text

0                                        didnt feel humili
1          feel hopeless damn hope around someon care awak
2                        grab minut post feel greedi wrong
3           ever feel nostalg fireplac know still properti
4                                             feel grouchi
                               ...                        
15357                                          feel suffer
15358    keep feel like someon unkind wrong think get b...
15359               feel littl cranki negat doctor appoint
15360      feel comfort derbi feel though start step shell
15361    feel weird meet peopl text like dont talk face...
Name: text, Length: 15362, dtype: object

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec=TfidfVectorizer(ngram_range=(1,2),max_features=20000,min_df=2,max_df=0.7)
data=vec.fit_transform(text)
data

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 169424 stored elements and shape (15362, 15812)>

In [53]:
y=df1.drop(['text'],axis=1)

In [54]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(data,y,test_size=0.30,random_state=42)

In [55]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
log=OneVsRestClassifier(LogisticRegression(C =0.5,max_iter=2000,class_weight='balanced'))
log.fit(x_train,y_train)
pred1=log.predict(x_test)

In [56]:
print(accuracy_score(y_test,pred1))

0.7711000216966805


In [57]:
from sklearn.metrics import f1_score

print("Micro F1:", f1_score(y_test, pred1, average='micro'))
print("Macro F1:", f1_score(y_test, pred1, average='macro'))


Micro F1: 0.858139534883721
Macro F1: 0.8405420019047757


In [58]:
prediction=log.predict(vec.transform(["i am feeling grouchy"]))
prediction

array([[0, 0, 0, 0, 0]])

In [59]:
from sklearn.svm import LinearSVC
model = OneVsRestClassifier(LinearSVC())
model.fit(x_train, y_train)

pred=model.predict(x_test)

c:\Users\VICTUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\VICTUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\VICTUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\VICTUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value

In [60]:
pred2=model.predict(vec.transform(["i am feeling grouchy"]))
pred2

array([[0, 0, 0, 0, 0]])

In [ ]:
def predict_emotion_with_confidence(text):
    
    prob = log.predict_proba(vec.transform([text]))
    prob_matrix = np.array(prob).reshape(1, -1)

    emotion_labels = [e for e in enc.categories_[0] if e != 'surprise']

    thresholds = {
        'anger': 0.25,
        'fear': 0.25,
        'joy': 0.5,
        'love': 0.3,
        'sadness': 0.5
    }

    pred = (prob_matrix >= np.array([thresholds[e] for e in emotion_labels])).astype(int)

    if pred.sum() == 0:
        pred[0][np.argmax(prob_matrix)] = 1

    predicted_emotions = []
    for i in range(len(pred[0])):
        if pred[0][i] == 1:
            predicted_emotions.append((emotion_labels[i], prob_matrix[0][i]))

    return predicted_emotions


In [62]:
predict_emotion_with_confidence("I am happy")


[('anger', 0.3069709265989033), ('fear', 0.27991374780675743)]

In [63]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier

# Wrap models in OneVsRestClassifier for multi-label support
knn = OneVsRestClassifier(KNeighborsClassifier())
sv  = OneVsRestClassifier(LinearSVC())
naive = OneVsRestClassifier(MultinomialNB())
dec = OneVsRestClassifier(DecisionTreeClassifier())
ran = OneVsRestClassifier(RandomForestClassifier())

models = [knn, sv, naive, dec, ran]
model_names = ['KNN', 'LinearSVC', 'MultinomialNB', 'DecisionTree', 'RandomForest']


In [64]:
from sklearn.metrics import f1_score

results = {}

for name, clf in zip(model_names, models):
    clf.fit(x_train, y_train)
    y_pred = clf.predict(x_test)
    
    micro = f1_score(y_test, y_pred, average='micro')
    macro = f1_score(y_test, y_pred, average='macro')
    
    results[name] = {'micro_f1': micro, 'macro_f1': macro}
    print(f"{name} => Micro F1: {micro:.4f}, Macro F1: {macro:.4f}")


KNN => Micro F1: 0.7119, Macro F1: 0.6744
LinearSVC => Micro F1: 0.8487, Macro F1: 0.8157


c:\Users\VICTUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\VICTUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\VICTUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\VICTUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value

MultinomialNB => Micro F1: 0.5505, Macro F1: 0.4103
DecisionTree => Micro F1: 0.8284, Macro F1: 0.8072
RandomForest => Micro F1: 0.8441, Macro F1: 0.8166


In [65]:
import joblib

joblib.dump(ran,'model.joblib')
joblib.dump(vec,'vectorizer.joblib')

['vectorizer.joblib']

In [66]:
MODEL_DIR = os.getcwd()
model=joblib.load(os.path.join(MODEL_DIR,'model.joblib'))
vectorizer=joblib.load(os.path.join(MODEL_DIR,'vectorizer.joblib'))